In [1]:
import os
import ast
import json
from tqdm import trange

import numpy as np
import pandas as pd

from scipy.stats import f_oneway, kruskal
from scipy import stats
from sklearn.metrics import normalized_mutual_info_score
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import OneHotEncoder

import scanpy as sc

import matplotlib.pyplot as plt
import seaborn as sns

import sys
sclembas = '/home/hmbaghda/Projects/scLEMBAS'
sys.path.insert(1, os.path.join(sclembas))
from scLEMBAS import io
from scLEMBAS import parse_network as pn
from scLEMBAS import preprocess as pp

/nobackup/users/hmbaghda/Software/miniforge3/envs/scLEMBAS/lib/python3.11/site-packages/anndata/utils.py:434: FutureWarning: Importing read_csv from `anndata` is deprecated. Import anndata.io.read_csv instead.
  warnings.warn(msg, FutureWarning)
/nobackup/users/hmbaghda/Software/miniforge3/envs/scLEMBAS/lib/python3.11/site-packages/anndata/utils.py:434: FutureWarning: Importing read_excel from `anndata` is deprecated. Import anndata.io.read_excel instead.
  warnings.warn(msg, FutureWarning)
/nobackup/users/hmbaghda/Software/miniforge3/envs/scLEMBAS/lib/python3.11/site-packages/anndata/utils.py:434: FutureWarning: Importing read_hdf from `anndata` is deprecated. Import anndata.io.read_hdf instead.
  warnings.warn(msg, FutureWarning)
/nobackup/users/hmbaghda/Software/miniforge3/envs/scLEMBAS/lib/python3.11/site-packages/anndata/utils.py:434: FutureWarning: Importing read_loom from `anndata` is deprecated. Import anndata.io.read_loom instead.
  warnings.warn(msg, FutureWarning)
/nobackup/


[KeOps] Warning : There were warnings or errors :
<stdin>:1:10: fatal error: cuda.h: No such file or directory
compilation terminated.

[KeOps] Warning : 
    The location of Cuda header files cuda.h and nvrtc.h could not be detected on your system.
    You must determine their location and then define the environment variable CUDA_PATH,
    either before launching Python or using os.environ before importing keops. For example
    if these files are in /vol/cuda/10.2.89-cudnn7.6.4.38/include you can do :
      import os
      os.environ['CUDA_PATH'] = '/vol/cuda/10.2.89-cudnn7.6.4.38'
    
[KeOps] Compiling cuda jit compiler engine ... 
[KeOps] Warning : There were warnings or errors :
/nobackup/users/hmbaghda/Software/miniforge3/envs/scLEMBAS/lib/python3.11/site-packages/keopscore/binders/nvrtc/nvrtc_jit.cpp:16:10: fatal error: cuda.h: No such file or directory
 #include <cuda.h>
          ^~~~~~~~
compilation terminated.

OK
[pyKeOps] Compiling nvrtc binder for python ... 
[KeOps] W

In [2]:
n_cores = 30
os.environ["OMP_NUM_THREADS"] = str(n_cores)
os.environ["MKL_NUM_THREADS"] = str(n_cores)
os.environ["OPENBLAS_NUM_THREADS"] = str(n_cores)
os.environ["VECLIB_MAXIMUM_THREADS"] = str(n_cores)
os.environ["NUMEXPR_NUM_THREADS"] = str(n_cores)

seed = 888
data_path = '/nobackup/users/hmbaghda/scLEMBAS/analysis'
author = 'Tahoe100M'

In [3]:
tf_adata = io.read_tfad(file_name = os.path.join(data_path, 'interim', author + 'consensus_tf_activity.h5ad'))
# need to run without the control
obs = tf_adata.obs.copy()
obs = obs[obs.drug != 'DMSO_TF']

# parse the drug target
obs.drug_target = obs.drug_target.astype(str).apply(lambda x: ast.literal_eval(x))

# Signaling Network Parsing

Load and parse input signaling network:

In [4]:
source_label = 'source_genesymbol'
target_label = 'target_genesymbol'
weight_label = 'mode_of_action'
stimulation_label = 'consensus_stimulation'
inhibition_label = 'consensus_inhibition'

In [5]:
sn_ppis = pn.load_network('omnipath', organism = 'human', static = True)
sn_ppis = pn.correct_network(sn_ppis = sn_ppis,
                                        source_label = source_label, target_label = target_label,
                                        stimulation_label = stimulation_label, inhibition_label = inhibition_label)

/home/hmbaghda/Projects/scLEMBAS/scLEMBAS/parse_network.py:162: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  unique_vals = pd.concat([unique_vals, dup_int], axis = 0)


Add the drug-target interactions:

In [6]:
url = "https://huggingface.co/datasets/tahoebio/Tahoe-100M/resolve/main/metadata/drug_metadata.parquet"
drug_ds = pd.read_parquet(url)

retained_drugs = obs.drug.unique()

moa_map = {'activator/agonist': 1, 
           'inhibitor/antagonist': -1}
drug_moa = dict(zip(drug_ds.drug, drug_ds['moa-broad']))
drug_moa = {k: moa_map[v] for k,v in drug_moa.items() if k in retained_drugs}
if len(drug_moa) != len(retained_drugs):
    raise ValueError('Missing drug moa')
    


In [7]:
delim = '&'
moa = []
interactions_to_add = []
for drug, drug_targets in dict(zip(obs.drug, obs.drug_target)).items():
    for drug_target in drug_targets:
        interactions_to_add.append(delim.join([drug, drug_target]))
        moa.append(drug_moa[drug])
        
if len(interactions_to_add) != len(set(interactions_to_add)):
    raise ValueError('Non-unique interactions are present')

In [8]:
sn_ppis = pn.add_omnipath_interaction(sn_ppis = sn_ppis,
                                      interactions_to_add = interactions_to_add,  
                                      moa = moa,
                                      delim = delim,
                                      source_label = source_label,
                                      target_label = target_label, 
                                      stimulation_label = stimulation_label, 
                                      inhibition_label = inhibition_label
                           )

Filtering the network for interactions with atleast one reference and a curation effort of 2: 

In [9]:
sn_ppis = pn.extract_network(sn_ppis = sn_ppis.copy(), 
                             curation_effort_thresh = 2, 
                             n_references_thresh = 1,
                             resources = 'all', 
                             drop_self = True, 
                             source_label = source_label, 
                             target_label = target_label,
                             verbose = False)

Retain a signaling network of nodes with full paths from input drugs to the inferred TFs:

In [10]:
ligand_labels = obs.drug.unique().tolist()
tf_labels = tf_adata.var.index.unique().tolist()

In [11]:
ppi_nodes = set(sn_ppis[source_label].tolist() + sn_ppis[target_label].tolist())
n_tf_intersect = len(ppi_nodes.intersection(tf_labels))
print('{} of {} TFs are present in Omnipath upon applying filters'.format(n_tf_intersect, 
                                                                          tf_adata.shape[1], 
                                                                         ))


534 of 716 TFs are present in Omnipath upon applying filters


In [12]:
sn_ppis, ligand_connections = pn.create_connected_network(sn_ppis = sn_ppis, 
                                                       ligand_labels = ligand_labels, 
                                                       tf_labels = tf_labels, 
                                                       source_label = source_label, 
                                                       target_label = target_label,
                                                       path_finder = 'shortest')

with open(os.path.join(data_path, 'processed', author + '_ligand_connections.json'), 'w') as f:
    json.dump(ligand_connections, f)

100%|████████████████████████████| 18156/18156 [00:01<00:00, 14853.46it/s]


In [13]:
ppi_nodes = set(sn_ppis[source_label].tolist() + sn_ppis[target_label].tolist())
retained_drugs = [drug for drug, TFs in ligand_connections.items() if len(TFs) != 0]
retained_TFs = ppi_nodes.intersection(tf_labels)
msg = 'The pruned network has {} interactions between {} nodes, '.format(sn_ppis.shape[0], len(ppi_nodes))
msg += 'retaining {} of {} TFs '.format(len(retained_TFs), n_tf_intersect)
msg += 'and {} of {} drugs (excluding control)'.format(len(retained_drugs), obs.drug.nunique())
print(msg)

The pruned network has 2728 interactions between 863 nodes, retaining 464 of 534 TFs and 28 of 34 drugs (excluding control)


Format network as needed for input to model building:

In [14]:
sn_ppis = pn.format_network(sn_ppis, weight_label, stimulation_label, inhibition_label) 

In [15]:
sn_ppis[[source_label, target_label, weight_label, stimulation_label, inhibition_label]].head()

,source_genesymbol,target_genesymbol,mode_of_action,consensus_stimulation,consensus_inhibition
0,MAPKAPK2,MAPK14,0.1,False,False
1,MAPK14,MAPKAPK2,1.0,True,False
2,NUMB,NOTCH1,-1.0,False,True
3,MAPK3,GABPA,1.0,True,False
4,MAPK1,GABPA,1.0,True,False


In [16]:
sn_ppis.to_csv(os.path.join(data_path, 'processed', author + '_sn_ppis.csv'))